<a href="https://colab.research.google.com/github/gmerveesilbir/uygulamalar/blob/main/Iris_KNN_SVM_Naive_Bayes_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Çiçek Türü Tahmini: KNN, SVM ve Naive Bayes
**Amaç:** Iris veri setini kullanarak üç sınıflandırma algoritmasını eğitmek, test etmek ve karşılaştırmak.


## 1. Kütüphaneleri ve veri setini yükleyelim
Iris: 150 gözlem, 4 sayısal özellik ve 3 çiçek türü.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
print("Çiçek sayısı:", len(X))
print("Özellikler:", list(X.columns))
print("Türler:", list(iris.target_names))
display(X.head())

**Tartışma:** Hangi sütunlar girdi (X)? Hangisi tahmin edilecek çıktı (y)?

## 2. Veriyi çizerek inceleyelim

In [ ]:
grafik_veri = X.copy()
grafik_veri["Tür"] = [iris.target_names[i] for i in y]
sns.scatterplot(data=grafik_veri, x="petal length (cm)", y="petal width (cm)", hue="Tür", s=65)
plt.title("Iris çiçekleri: taç yaprak ölçüleri")
plt.show()

**Soru:** Hangi türün diğerlerinden daha kolay ayrıldığını düşünüyorsunuz?

## 3. Eğitim ve test verilerine ayıralım
Verilerin %80’i eğitim, %20’si test için. `stratify=y` her türün oranını korur.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Eğitim örneği:", len(X_train))
print("Test örneği:", len(X_test))

## 4. KNN ile ilk modeli kuralım
`n_neighbors=3` en yakın 3 komşuya bakılması demektir. `StandardScaler` değişkenlerin ölçeğini dengeler. Pipeline yalnızca eğitim verilerinden ölçekleme öğrenir.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
knn.fit(X_train, y_train)
knn_tahmin = knn.predict(X_test)
print("KNN doğruluk:", round(accuracy_score(y_test, knn_tahmin)*100, 2), "%")

## 5. SVM ve Naive Bayes deneyelim
SVM için de ölçekleme kullanıyoruz; GaussianNB sayısal sürekli veriler için uygundur.

In [ ]:
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
nb = GaussianNB()
svm.fit(X_train, y_train)
nb.fit(X_train, y_train)
svm_tahmin = svm.predict(X_test)
nb_tahmin = nb.predict(X_test)
print("SVM doğruluk:", round(accuracy_score(y_test, svm_tahmin)*100, 2), "%")
print("Naive Bayes doğruluk:", round(accuracy_score(y_test, nb_tahmin)*100, 2), "%")

## 6. Modelleri karşılaştıralım

In [ ]:
sonuclar = pd.DataFrame({
    "Algoritma": ["KNN", "SVM", "Naive Bayes"],
    "Doğruluk (%)": [accuracy_score(y_test, knn_tahmin)*100,
                      accuracy_score(y_test, svm_tahmin)*100,
                      accuracy_score(y_test, nb_tahmin)*100]
})
display(sonuclar.round(2))
plt.bar(sonuclar["Algoritma"], sonuclar["Doğruluk (%)"])
plt.ylim(0, 105)
plt.ylabel("Test doğruluğu (%)")
plt.title("Sınıflandırma algoritmalarının karşılaştırılması")
plt.show()

## 7. Hangi türler karıştırılıyor? (KNN hata matrisi)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, knn_tahmin, display_labels=iris.target_names, cmap="Blues")
plt.title("KNN hata matrisi")
plt.show()

## 8. Yeni bir çiçek tahmin edelim
Özellik sırası: sepal length, sepal width, petal length, petal width. Gerçek bir çiçeğin ölçümünü girebilirsiniz. Bu örnek yapay bir ölçümdür.

In [ ]:
yeni_cicek = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=X.columns)
for ad, model in [("KNN", knn), ("SVM", svm), ("Naive Bayes", nb)]:
    tahmin = model.predict(yeni_cicek)[0]
    print(ad, "tahmini:", iris.target_names[tahmin])

## 9. Öğrenci deneyleri
1. KNN hücresindeki `n_neighbors=3` değerini sırasıyla 1, 5 ve 9 yapın. Değişimleri not edin.
2. Eğitim-test ayrımındaki `test_size=0.20` değerini `0.30` yapın. **3. hücreden sonraki tüm hücreleri yeniden çalıştırın.** Sonuçları karşılaştırın.
3. `yeni_cicek` ölçümlerini değiştirip hangi türün tahmin edildiğini gözlemleyin.
4. Tek bir test bölmesindeki yüksek doğruluk, her veri kümesinde aynı sonucu garanti eder mi? Tartışın.

